In [1]:
# 1. Ma'lumotni yuklash
import re
from collections import Counter

import pandas as pd
from datasets import load_dataset

import importlib
import normalize
importlib.reload(normalize)

df = pd.read_csv("../03_trim/trim_manifest_yakuniy.csv")

# raw_text kerak: undan "M.Yusupova" va "basqa da" ni tiklaymiz
ds = load_dataset("atikuwu/karakalpak-speech-corpus", cache_dir="../hf_cache")
raw = {}
for split in ds:
    raw[split] = ds[split]["raw_text"]

df["raw_text"] = [raw[s][i] for s, i in zip(df["split"], df["idx"])]
print("Matnlar:", len(df))

<python>\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Matnlar: 21470


In [2]:
# 2. Kam uchraydigan belgilar
belgilar = Counter("".join(df["matn"]))

for belgi, soni in sorted(belgilar.items(), key=lambda x: x[1]):
    if soni < 1000:
        print(repr(belgi), soni)

'ỹ' 1
'ȳ' 1
"'" 1
'_' 7
'é' 7
'Ú' 220
'Ǵ' 259
'!' 260
'?' 298
'W' 445
'Z' 452
'(' 486
'L' 488
'C' 551
')' 640
';' 670
'V' 713
'Y' 774
':' 938
'G' 978


In [3]:
# 3. Muammolarni sanash
tekshiruv = {
    "raqam": r"\d",
    "kirill harf": r"[а-яА-ЯўЎқҚғҒҳҲ]",
    "ȳ ỹ": r"[ȳỹ]",
    "ÓzA oxirida": r"ÓzA\W*$",
    "rim artefakti": r"\b(?:mıńınshı|birinshi|besinshi|onınshı|eliwinshi|júzinshi)\.\s+[A-Z]",
    "birliklar": r"\b(?:mln|mlrd|trln|kg|km|kVt)\b",
    "qavs": r"\(",
}

for nom, naqsh in tekshiruv.items():
    print(nom, df["matn"].str.contains(naqsh).sum())

raqam 0
kirill harf 0
ȳ ỹ 2
ÓzA oxirida 517
rim artefakti 209
birliklar 223
qavs 416


In [4]:
# 4. Normalizatsiya
df["matn_model"] = [normalize.model_matni(t, r) for t, r in zip(df["matn"], df["raw_text"])]

for i in df.sample(5, random_state=0).index:
    print("ESKI :", df.loc[i, "matn"])
    print("YANGI:", df.loc[i, "matn_model"])
    print()

ESKI : Egin egiw jolların úyret dep kelgen jaslarǵa bárqulla járdem qolın sozıp, nállerin berip, olardı qalay tárbiyalaw kerekligin úyretiwdende jalıqpaytuǵın Durdıniyaz aǵanıń kóklemzarlıq qushaǵanda jaylasqan shańaraǵınıń biz bir mezgillik turmısın ǵana kórdik. "Miynet etseń erinbey, toyadı qarnıń tilenbey",-degen sózdiń qanshelli haqıyqat ekenligi biziń kóz aldımızda óz dálili menen sáwlelengen miynetkesh adamlardıń awızbirshiligine,
YANGI: egin egiw jolların úyret dep kelgen jaslarǵa bárqulla járdem qolın sozıp nállerin berip olardı qalay tárbiyalaw kerekligin úyretiwden de jalıqpaytuǵın durdıniyaz aǵanıń kóklemzarlıq qushaǵanda jaylasqan shańaraǵınıń biz bir mezgillik turmısın ǵana kórdik miynet etseń erinbey toyadı qarnıń tilenbey degen sózdiń qanshelli haqıyqat ekenligi biziń kóz aldımızda óz dálili menen sáwlelengen miynetkesh adamlardıń awızbirshiligine

ESKI : Nátiyjede sud qolǵa alıw ámeliyatın nızamsız, dep tapqan. Advokat Sok Don Xyonnıń túsindiriwinshe, puqaralardıń appel

In [5]:
# 5. Qavsli matnlarni chiqarish va saqlash
# Qavs ichidagi matn ba'zan o'qiladi, ba'zan yo'q -> ishonchsiz
qavsli = df["matn"].str.contains(r"\(")
final = df[~qavsli]

print("Qavs sababli chiqarildi:", qavsli.sum())
print("Qoldi:", len(final))
print("Soat:", round(final["yangi_s"].sum() / 3600, 2))
print(final.groupby("split").size())

# Tekshiruv: faqat kichik harflar va probel qolishi kerak
print("Belgilar:", "".join(sorted(set("".join(final["matn_model"])))))

ustunlar = ["split", "idx", "fayl", "yangi_s", "matn", "matn_model"]
df = final[ustunlar].copy()   # oraliq natija, alohida CSV qilib saqlanmaydi

Qavs sababli chiqarildi: 416
Qoldi: 21054
Soat: 101.49
split
test           1029
train         18980
validation     1045
dtype: int64
Belgilar:  abcdefghijklmnopqrstuvwxyzáéóúıńǵ


In [6]:
# 6. 30 s dan uzun audiolarni chiqarish (Whisper 30 s dan ortig'ini qabul qilmaydi)

uzun = df["yangi_s"] > 30
print("30 s dan uzun:", uzun.sum(), "| soat:", round(df.loc[uzun, "yangi_s"].sum() / 3600, 2))
df = df[~uzun]

30 s dan uzun: 236 | soat: 2.09


In [7]:
# 7. Leakage: val/test matni train'da ham bo'lsa, train'dan chiqaramiz
valtest_matnlar = set(df.loc[df["split"] != "train", "matn_model"])

leak = (df["split"] == "train") & df["matn_model"].isin(valtest_matnlar)
print("Train'dagi val/test matnlari:", leak.sum())
df = df[~leak]

# Validation va test o'rtasida ham tekshiramiz
val_test = set(df.loc[df["split"] == "validation", "matn_model"]) & set(df.loc[df["split"] == "test", "matn_model"])
print("Validation va test o'rtasida:", len(val_test))
df = df[~((df["split"] == "validation") & df["matn_model"].isin(val_test))]

Train'dagi val/test matnlari: 15
Validation va test o'rtasida: 1


In [8]:
# 8. Bitta split ichidagi dublikatlar: birinchisi qoladi
dublikat = df.duplicated(["split", "matn_model"], keep="first")
print("Dublikatlar:", dublikat.sum())
print(df[dublikat].groupby("split").size())
df = df[~dublikat]

print("\nQoldi:", len(df), "| soat:", round(df["yangi_s"].sum() / 3600, 2))
print(df.groupby("split").size())
df.to_csv("manifest_tayyor.csv", index=False)

Dublikatlar: 65
split
train    65
dtype: int64

Qoldi: 20737 | soat: 99.01
split
test           1018
train         18685
validation     1034
dtype: int64
